In [1]:
import pandas as pd
import numpy as np
import warnings
import re
warnings.filterwarnings('ignore')

# missing_outlier_handling_v3.ipynb 실행 후 저장된 clean 파일 로드
train = pd.read_csv('train_clean.csv')
test  = pd.read_csv('test_clean.csv')

TARGET = '임신 성공 여부'
ID_COL = 'ID'

is_di_train = train['시술 유형'] == 'DI'
is_di_test  = test['시술 유형']  == 'DI'

print('Train:', train.shape, '/ Test:', test.shape)
print(f'IVF: {(~is_di_train).sum()}건 ({(~is_di_train).mean()*100:.1f}%)')
print(f'DI : {is_di_train.sum()}건 ({is_di_train.mean()*100:.1f}%)')

Train: (256351, 89) / Test: (90067, 88)
IVF: 250060건 (97.5%)
DI : 6291건 (2.5%)


In [2]:
# 전처리: 횟수 컬럼 정수 변환
COUNT_MAP = {'0회':0,'1회':1,'2회':2,'3회':3,'4회':4,'5회':5,'6회 이상':6}
COUNT_COLS = [
    '총 시술 횟수','클리닉 내 총 시술 횟수',
    'IVF 시술 횟수','DI 시술 횟수',
    '총 임신 횟수','IVF 임신 횟수','DI 임신 횟수',
    '총 출산 횟수','IVF 출산 횟수','DI 출산 횟수',
]
for df in [train, test]:
    for col in COUNT_COLS:
        if col in df.columns and df[col].dtype == 'object':
            df[col] = df[col].map(COUNT_MAP)
print('횟수 변환 완료')

횟수 변환 완료


In [3]:
# 나이 피처
AGE_MAP = {
    '만18-34세':1,'만35-37세':2,'만38-39세':3,
    '만40-42세':4,'만43-44세':5,'만45-50세':6,'알 수 없음':np.nan
}
# EDA 실측 성공률 기반 사전 점수 (도메인 priori — train 통계 아님, leakage 없음)
AGE_PRIOR_RATE = {
    '만18-34세':0.32,'만35-37세':0.27,'만38-39세':0.22,
    '만40-42세':0.16,'만43-44세':0.12,'만45-50세':0.17,'알 수 없음':np.nan
}

for df in [train, test]:
    df['나이_순서']           = df['시술 당시 나이'].map(AGE_MAP)
    df['나이_성공률_사전점수'] = df['시술 당시 나이'].map(AGE_PRIOR_RATE)

    # 임상 임계 플래그
    df['나이_35이상'] = (df['나이_순서'] >= 2).astype(int)  # 난자 질 저하 시작
    df['나이_38이상'] = (df['나이_순서'] >= 3).astype(int)  # IVF 강력 권고
    df['나이_40이상'] = (df['나이_순서'] >= 4).astype(int)  # 이수성 배아 50%↑
    df['나이_43이상'] = (df['나이_순서'] >= 5).astype(int)  # 자가 난자 한계

print('나이 피처 완료')

나이 피처 완료


In [4]:
# 난소 예비능 프록시 피처
for df in [train, test]:
    egg_n = df['수집된 신선 난자 수'].fillna(0)

    # 2-1. 난소 반응 그룹 (POSEIDON 기준)
    df['난소반응_그룹'] = pd.cut(
        egg_n,
        bins=[-1, 0, 3, 14, 100],
        labels=['채취없음', 'Poor(1-3개)', 'Normal(4-14개)', 'High(15개이상)']
    ).astype(str)

    # 2-2. Poor Responder 플래그 (성공률 저하 위험)
    df['poor_responder'] = ((egg_n > 0) & (egg_n < 4)).astype(int)

    # 2-3. High Responder 플래그 (OHSS 위험, Freeze-all 가능성)
    df['high_responder'] = (egg_n >= 15).astype(int)

    # 2-4. Freeze-all 전략 탐지
    # 신선 이식 없음 + 배아 저장 있음 → Freeze-all 전략
    fresh_tx = df['신선 배아 사용 여부'].fillna(0).astype(int)
    stored   = df['저장된 배아 수'].fillna(0)
    df['freeze_all_전략'] = ((fresh_tx == 0) & (stored > 0)).astype(int)

    # 2-5. High Responder + Freeze-all 조합 (OHSS 예방 목적 강하게 시사)
    df['high_resp_freezeall'] = (
        (df['high_responder'] == 1) & (df['freeze_all_전략'] == 1)
    ).astype(int)

    # 2-6. log1p 변환 (난자 수 Long-tail 해소)
    df['난자수_log'] = np.log1p(egg_n)

print('난소 예비능 피처 완료')
print(train['난소반응_그룹'].value_counts())

난소 예비능 피처 완료
난소반응_그룹
Normal(4-14개)    128081
채취없음              60136
High(15개이상)       48690
Poor(1-3개)        19444
Name: count, dtype: int64


In [5]:
# 배아 품질 프록시 점수
def safe_ratio(num, denom):
    """분모 0 → 0 반환 (도메인 근거: 단계 미진입 = 0)"""
    return np.where(denom > 0, num / denom, 0)

for df in [train, test]:
    # 3-1. 핵심 효율 비율 (다중공선성 해소)
    df['수정_효율']          = safe_ratio(df['총 생성 배아 수'],              df['혼합된 난자 수'])
    df['ICSI_수정_효율']     = safe_ratio(df['미세주입에서 생성된 배아 수'],   df['미세주입된 난자 수'])
    df['이식_효율']          = safe_ratio(df['이식된 배아 수'],               df['총 생성 배아 수'])
    df['저장_비율']          = safe_ratio(df['저장된 배아 수'],               df['총 생성 배아 수'])
    df['난자_배아_전환율']   = safe_ratio(df['총 생성 배아 수'],              df['수집된 신선 난자 수'])
    df['파트너정자_비율']    = safe_ratio(df['파트너 정자와 혼합된 난자 수'], df['혼합된 난자 수'])

    # 3-2. 배아 종합 품질 점수 (3가지 효율 가중 합산)
    # 임상적으로 수정 40% + 이식 40% + 저장 20% 가중치 (배반포 전환 대리)
    df['배아품질_종합점수'] = (
        df['수정_효율'] * 0.4 +
        df['이식_효율'] * 0.4 +
        df['저장_비율'] * 0.2
    )

    # 3-3. log1p 변환 (Long-tail 해소, 상관 높은 군 대표 1개씩)
    for col in ['총 생성 배아 수','이식된 배아 수','수집된 신선 난자 수',
                '혼합된 난자 수','저장된 배아 수']:
        if col in df.columns:
            df[f'{col}_log'] = np.log1p(df[col].fillna(0))

    # 3-4. Binary 진입여부 (Zero-inflation 보완)
    for flag, src in [
        ('배아생성_있음','총 생성 배아 수'),
        ('이식배아_있음','이식된 배아 수'),
        ('배아저장_있음','저장된 배아 수'),
        ('ICSI시술_있음','미세주입된 난자 수'),
        ('해동배아_있음','해동된 배아 수'),
    ]:
        if src in df.columns:
            df[flag] = (df[src].fillna(0) > 0).astype(int)

    # 3-5. 이식 배아 수 구간화
    df['이식배아_구간'] = pd.cut(
        df['이식된 배아 수'].fillna(0),
        bins=[-1,0,1,2,100], labels=['0개','1개','2개','3개이상']
    ).astype(str)

    # 3-6. 잔여 배아 수 (이식 후 저장 가능 배아)
    df['잔여배아_수'] = (df['총 생성 배아 수'].fillna(0) - df['이식된 배아 수'].fillna(0)).clip(lower=0)

    # 3-7. 비율 [0,1] 클리핑
    for col in ['수정_효율','ICSI_수정_효율','이식_효율','저장_비율',
                '난자_배아_전환율','파트너정자_비율','배아품질_종합점수']:
        df[col] = df[col].clip(0, 1)

print('배아 품질 피처 완료')

배아 품질 피처 완료


In [6]:
# 시술 유형 파싱 + 배아 이식 일수(D3/D5)
for df in [train, test]:
    proc = df['특정 시술 유형'].fillna('').str.upper()

    df['배반포이식_여부']  = proc.str.contains('BLASTOCYST').astype(int)
    df['ICSI_여부']        = proc.str.contains('ICSI').astype(int)
    df['FER_여부']         = proc.str.contains('FER').astype(int)
    df['AH_여부']          = proc.str.contains(r'\bAH\b').astype(int)
    df['IUI_여부']         = proc.str.contains('IUI').astype(int)
    df['복합시술_여부']    = proc.str.contains(r'[/:]').astype(int)

    # 임상 근거 기반 고성공 조합
    df['ICSI_배반포_조합'] = ((df['ICSI_여부']==1) & (df['배반포이식_여부']==1)).astype(int)
    df['IVF_배반포_조합']  = ((df['시술 유형']=='IVF') & (df['배반포이식_여부']==1)).astype(int)

    # 기술 집약도 점수
    tech_cols = ['ICSI_여부','배반포이식_여부','AH_여부',
                 '착상 전 유전 검사 사용 여부','착상 전 유전 진단 사용 여부']
    df['기술집약도_점수'] = df[[c for c in tech_cols if c in df.columns]].fillna(0).sum(axis=1)

# Rare category 병합 (빈도 50건 미만 → OTHER)
# EDA: 성공률 1.0 카테고리는 표본 극소수 → 과적합 위험
proc_freq  = train['특정 시술 유형'].value_counts()
rare_proc  = proc_freq[proc_freq < 50].index
stim_freq  = train['배란 유도 유형'].value_counts()
rare_stim  = stim_freq[stim_freq < 50].index

for df in [train, test]:
    df['특정시술_정제'] = df['특정 시술 유형'].apply(
        lambda x: 'OTHER' if x in rare_proc else (x if pd.notna(x) else 'missing')
    )
    df['배란유도_정제'] = df['배란 유도 유형'].apply(
        lambda x: 'OTHER' if x in rare_stim else (x if pd.notna(x) else 'missing')
    )

print('시술 유형 피처 완료')
print(f'Rare 처리 특정시술: {len(rare_proc)}개 / 배란유도: {len(rare_stim)}개 → OTHER')

시술 유형 피처 완료
Rare 처리 특정시술: 8개 / 배란유도: 2개 → OTHER


In [7]:
# 경과일 간격 + 배아 이식 일수(D3/D5) 분류
for df in [train, test]:
    is_di = df['시술 유형'] == 'DI'

    # 배양 기간 (혼합→이식): D3 vs D5 핵심
    df['배양_기간_일']    = df['배아 이식 경과일'] - df['난자 혼합 경과일']
    df['채취_이식_총기간'] = df['배아 이식 경과일'] - df['난자 채취 경과일']
    df['채취_혼합_간격']  = df['난자 혼합 경과일'] - df['난자 채취 경과일']
    df['해동_이식_간격']  = df['배아 이식 경과일'] - df['배아 해동 경과일']

    # DI는 -1, IVF 음수 오류 → NaN
    for col in ['배양_기간_일','채취_이식_총기간','채취_혼합_간격','해동_이식_간격']:
        df.loc[is_di, col] = df.loc[is_di, col].fillna(-1)
        df.loc[~is_di & (df[col] < 0), col] = np.nan

    # D3 / D5 / 기타 / DI 분류
    def classify_day(row):
        if row['시술 유형'] == 'DI':
            return 'DI_NA'
        d = row['배양_기간_일']
        if pd.isna(d) or d < 0:
            return '미상'
        if d <= 3.5:
            return 'D3_세포기'
        elif d <= 4.5:
            return 'D4_모룰라'
        else:
            return 'D5이상_배반포'

    df['이식_단계_범주'] = df.apply(classify_day, axis=1)

    # D5 배반포 추정 플래그
    df['배양기간_D5추정'] = (df['이식_단계_범주'] == 'D5이상_배반포').astype(int)

print('경과일 피처 완료')
print(train['이식_단계_범주'].value_counts())


경과일 피처 완료
이식_단계_범주
D3_세포기      89240
D5이상_배반포    83902
미상          74266
DI_NA        6291
D4_모룰라       2652
Name: count, dtype: int64


In [8]:
# 반복 착상 실패(RIF) 및 시술 이력 피처
# 숫자가 포함된 문자열에서 숫자만 뽑아내는 함수
def extract_number(series):
    # '1회' -> 1, '6회 이상' -> 6, '결측/알수없음' -> NaN
    return pd.to_numeric(series.astype(str).str.extract(r'(\d+)')[0], errors='coerce')

def bin_count(series):
    # 이미 전처리된 수치형 시리즈를 받아서 버킷팅
    return pd.cut(
        series.fillna(-1),
        bins=[-2, -0.5, 0.5, 1.5, 2.5, 3.5, 100],
        labels=['결측', '0회', '1회', '2회', '3회', '4회이상']
    ).astype(str)

for df in [train, test]:
    # --- 전처리: 문자열 컬럼을 수치형으로 변환 ---
    # 변환이 필요한 주요 횟수 컬럼들
    count_cols = ['총 시술 횟수', 'IVF 시술 횟수', '총 임신 횟수', 'IVF 임신 횟수', '총 출산 횟수', 'IVF 출산 횟수', '클리닉 내 총 시술 횟수']
    
    for col in count_cols:
        if col in df.columns:
            df[col] = extract_number(df[col])

    # 6-1. 횟수 버킷팅 (4회↑ 묶기)
    df['총시술_버킷']  = bin_count(df['총 시술 횟수'])
    df['IVF시술_버킷'] = bin_count(df['IVF 시술 횟수'])

    # 6-2. 실패 횟수
    df['총_실패_횟수']  = (df['총 시술 횟수'] - df['총 임신 횟수']).clip(lower=0)
    df['IVF_실패_횟수'] = (df['IVF 시술 횟수'] - df['IVF 임신 횟수']).clip(lower=0)

    # 6-3. RIF 플래그 (반복 착상 실패: 3회↑ 이식 & 임신 0)
    df['RIF_플래그'] = (
        (df['총 시술 횟수'] >= 3) & (df['총 임신 횟수'] == 0)
    ).astype(int)

    # 6-4. RIF + AH 사용 조합 (AH는 RIF 환자에서 주로 사용)
    df['RIF_AH_조합'] = ((df['RIF_플래그']==1) & (df['AH_여부']==1)).astype(int)

    # 6-5. RIF + PGT 사용 조합 (배아 염색체 선별 목적)
    pgt_used = (
        df.get('착상 전 유전 검사 사용 여부', pd.Series(0, index=df.index)).fillna(0) +
        df.get('착상 전 유전 진단 사용 여부', pd.Series(0, index=df.index)).fillna(0)
    ) > 0
    df['RIF_PGT_조합'] = ((df['RIF_플래그']==1) & pgt_used).astype(int)

    # 6-6. 첫 시도 여부
    df['첫시도_여부'] = (df['총 시술 횟수'] == 0).astype(int)

    # 6-7. 과거 임신 성공률 (분모 0 → NaN, 0으로 채우면 편향)
    df['과거_임신성공률'] = np.where(
        df['총 시술 횟수'] > 0,
        df['총 임신 횟수'] / df['총 시술 횟수'], np.nan
    )
    df['임신_출산전환율'] = np.where(
        df['총 임신 횟수'] > 0,
        df['총 출산 횟수'] / df['총 임신 횟수'], np.nan
    )

    # 6-8. 출산 경험 플래그 (CLBR: 생식 능력 입증 — 가장 강한 양성 예측 인자)
    df['출산경험_있음'] = (df['총 출산 횟수'] > 0).astype(int)
    df['IVF출산경험_있음'] = (df['IVF 출산 횟수'] > 0).astype(int)

    # 6-9. 클리닉 충성도 (동일 클리닉 반복 = 신뢰/프로토콜 연속성)
    df['클리닉_시술비율'] = np.where(
        df['총 시술 횟수'] > 0,
        df['클리닉 내 총 시술 횟수'] / df['총 시술 횟수'], np.nan
    )

print('시술 이력 피처 완료')

시술 이력 피처 완료


In [9]:
# 불임 원인 조합 피처
cause_cols = [
    '불임 원인 - 난관 질환','불임 원인 - 남성 요인','불임 원인 - 배란 장애',
    '불임 원인 - 여성 요인','불임 원인 - 자궁경부 문제','불임 원인 - 자궁내막증',
    '불임 원인 - 정자 농도','불임 원인 - 정자 면역학적 요인',
    '불임 원인 - 정자 운동성','불임 원인 - 정자 형태'
]
cause_cols  = [c for c in cause_cols if c in train.columns]
sperm_cols  = [c for c in cause_cols if '정자' in c]
female_cols = [
    '불임 원인 - 난관 질환','불임 원인 - 배란 장애',
    '불임 원인 - 여성 요인','불임 원인 - 자궁경부 문제','불임 원인 - 자궁내막증'
]
female_cols = [c for c in female_cols if c in train.columns]

for df in [train, test]:
    df['불임원인_수']     = df[cause_cols].fillna(0).sum(axis=1)
    df['정자_문제_수']    = df[sperm_cols].fillna(0).sum(axis=1)
    df['여성_원인_수']    = df[female_cols].fillna(0).sum(axis=1)

    # 중증 남성 불임: 정자 3가지 이상 동시 문제 (TESE/ICSI 적응증)
    df['중증남성불임_플래그'] = (df['정자_문제_수'] >= 3).astype(int)

    # 자궁내막증 플래그 (난소 예비능 저하 가능)
    endo_col = '불임 원인 - 자궁내막증'
    if endo_col in df.columns:
        df['자궁내막증_플래그'] = df[endo_col].fillna(0).astype(int)
        # 자궁내막증 + Poor Responder 조합 (난소 손상 가능성)
        df['자궁내막증_poor_resp'] = (
            (df['자궁내막증_플래그']==1) & (df['poor_responder']==1)
        ).astype(int)

    # 복합 원인 플래그
    df['복합원인_플래그'] = (df['불임원인_수'] >= 2).astype(int)
    df['원인불명_플래그'] = df['불명확 불임 원인'].fillna(0).astype(int)

    # 원인 주체 분류
    has_male   = (df['남성 주 불임 원인'].fillna(0)==1) | (df['정자_문제_수']>0)
    has_female = (df['여성 주 불임 원인'].fillna(0)==1) | (df['여성_원인_수']>0)
    has_both   = df['부부 주 불임 원인'].fillna(0)==1
    df['불임원인_주체'] = np.select(
        [has_both, has_male & ~has_female, has_female & ~has_male,
         has_male & has_female, df['원인불명_플래그'].astype(bool)],
        ['부부복합','남성단독','여성단독','남녀복합','불명확'],
        default='기타'
    )

    # IUI 최적 대상 조합: 원인불명 + DI + 나이 < 35
    is_di = df['시술 유형'] == 'DI'
    df['IUI최적_조건'] = (
        (df['원인불명_플래그']==1) & is_di & (df['나이_35이상']==0)
    ).astype(int)

print('불임 원인 피처 완료')
print(train['불임원인_주체'].value_counts())

불임 원인 피처 완료
불임원인_주체
기타      111965
여성단독     76723
불명확      58994
부부복합      8477
남녀복합       109
남성단독        83
Name: count, dtype: int64


In [10]:
# PGT(착상 전 유전 검사) 관련 피처
PGS_COL = '착상 전 유전 검사 사용 여부'
PGD_COL = '착상 전 유전 진단 사용 여부'

for df in [train, test]:
    pgs = df.get(PGS_COL, pd.Series(0, index=df.index)).fillna(0)
    pgd = df.get(PGD_COL, pd.Series(0, index=df.index)).fillna(0)

    df['PGT_사용_여부'] = ((pgs > 0) | (pgd > 0)).astype(int)

    # 고령 + PGT: 38세↑에서 PGT 효과 극대화
    df['고령_PGT_조합'] = ((df['나이_38이상']==1) & (df['PGT_사용_여부']==1)).astype(int)

    # RIF + PGT (반복 실패 환자의 염색체 검사)
    df['RIF_PGT_조합'] = ((df['RIF_플래그']==1) & (df['PGT_사용_여부']==1)).astype(int)

    # 단일 배아 이식 + PGT (검사 후 최상 배아 1개만 선택)
    single_tx = df.get('단일 배아 이식 여부', pd.Series(0, index=df.index)).fillna(0)
    df['PGT_단일이식_조합'] = ((df['PGT_사용_여부']==1) & (single_tx==1)).astype(int)

print('PGT 피처 완료')

PGT 피처 완료


In [11]:
# 난자 정자 출처 피처
DONOR_AGE_MAP = {
    '만20세 이하':1,'만21-25세':2,'만26-30세':3,
    '만31-35세':4,'만36-40세':5,'만41-45세':6,'알 수 없음':np.nan
}
for df in [train, test]:
    df['난자기증자_나이_순서'] = df['난자 기증자 나이'].map(DONOR_AGE_MAP)
    df['기증난자_사용']        = (df['난자 출처'] == '기증 제공').astype(int)
    df['기증정자_사용']        = (df['정자 출처'] == '기증 제공').astype(int)

    # 자가 난자 + 40세↑ (이수성 배아 비율 50%↑)
    df['자가난자_고령_조합'] = (
        (df['난자 출처'] == '본인 제공') & (df['나이_40이상']==1)
    ).astype(int)

    # 자가 난자 + 43세↑ (자가 난자 임상 한계)
    df['자가난자_한계연령'] = (
        (df['난자 출처'] == '본인 제공') & (df['나이_43이상']==1)
    ).astype(int)

    # 젊은 기증 난자 (기증자 나이 ≤ 30세: 1~3)
    df['젊은기증난자_조합'] = (
        (df['기증난자_사용']==1) &
        (df['난자기증자_나이_순서'].fillna(99) <= 3)
    ).astype(int)

    # 대리모 여부 (완전히 다른 임신 메커니즘)
    if '대리모 여부' in df.columns:
        df['대리모_플래그'] = df['대리모 여부'].fillna(0).astype(int)

print('출처 피처 완료')

출처 피처 완료


In [12]:
# 핵심 상호작용 피처
for df in [train, test]:
    is_ivf = (df['시술 유형'] == 'IVF').astype(int)
    age    = df['나이_순서'].fillna(0)
    tx_cnt = df['총 시술 횟수'].fillna(0)
    trial  = df.get('임신 시도 또는 마지막 임신 경과 연수', pd.Series(0, index=df.index)).fillna(0)

    # 나이 × IVF (EDA 1순위 + 도메인 일치)
    df['나이_IVF_상호작용']       = age * is_ivf
    # 나이 × 배반포 이식 (고령일수록 배반포 선별 효과↑)
    df['나이_배반포_상호작용']     = age * df['배반포이식_여부']
    # 나이 × 배아 품질 점수 (핵심: 고령 + 낮은 품질 = 최악)
    df['나이_배아품질_상호작용']   = age * df['배아품질_종합점수']
    # 나이 × 반복 실패 (고령 + 반복 = 난임 심화)
    df['나이_시술횟수_상호작용']   = age * tx_cnt
    # Poor Responder × 수정 효율 (채취 적은데 수정도 안됨)
    df['poor_resp_수정효율']       = df['poor_responder'] * df['수정_효율']
    # 출산 경험 × 현재 나이 (성공 경험 있지만 나이 들면서 어려워짐)
    df['출산경험_나이_상호작용']   = df['출산경험_있음'] * age
    # 3중 고위험 (40세↑ + 자가 난자 + IVF)
    df['고위험_3중_조합'] = (
        (df['나이_40이상']==1) & (df['기증난자_사용']==0) & (is_ivf==1)
    ).astype(int)
    # 난임 종합 난이도 점수
    df['난임_난이도_점수'] = (
        age * 0.4 +
        df['총_실패_횟수'].fillna(0) * 0.3 +
        trial * 0.15 +
        df['복합원인_플래그'].fillna(0) * 0.15
    )
    # 시술 성공 잠재력 점수 (긍정 인자 합산)
    df['성공잠재력_점수'] = (
        df['배아품질_종합점수'] * 0.4 +
        df['과거_임신성공률'].fillna(0) * 0.3 +
        df['기술집약도_점수'] / 5 * 0.2 +
        df['출산경험_있음'] * 0.1
    )

print('상호작용 피처 완료')

상호작용 피처 완료


In [13]:
# 배아 생성 목적 + 고결측 변수 처리
# ── 11-1. 배아 생성 목적 (EDA: 현재시술용 포함 그룹 성공률↑) ─────
for df in [train, test]:
    reason = df['배아 생성 주요 이유'].fillna('').str
    df['현재시술용_포함'] = reason.contains('현재 시술용').astype(int)
    df['배아저장용_포함'] = reason.contains('배아 저장용').astype(int)
    df['난자저장용_포함'] = reason.contains('난자 저장용').astype(int)
    df['기증용_포함']     = reason.contains('기증용').astype(int)
    df['목적_수'] = (
        df['현재시술용_포함'] + df['배아저장용_포함'] +
        df['난자저장용_포함'] + df['기증용_포함']
    )
    # 즉시 임신 단독 목적 (저장 없음)
    df['즉시임신단독_여부'] = (
        (df['현재시술용_포함']==1) &
        (df['배아저장용_포함']==0) &
        (df['난자저장용_포함']==0)
    ).astype(int)

# ── 11-2. 고결측 변수 → 결측 여부 플래그만 보존 ───────────────────
# EDA: 96~99% 결측 → 결측 여부 자체가 정보
high_miss = [
    '임신 시도 또는 마지막 임신 경과 연수',
    '착상 전 유전 검사 사용 여부',
    '착상 전 유전 진단 사용 여부',
    'PGS 시술 여부', 'PGD 시술 여부',
    '난자 해동 경과일',
]
high_miss = [c for c in high_miss if c in train.columns]
for col in high_miss:
    rate = train[col].isnull().mean()
    for df in [train, test]:
        df[f'{col}_결측여부'] = df[col].isnull().astype(int)
    print(f'  {col}: {rate*100:.1f}% 결측 → 플래그 생성')

# ── 11-3. 장기 난임 피처 ──────────────────────────────────────────
COL_YEARS = '임신 시도 또는 마지막 임신 경과 연수'
if COL_YEARS in train.columns:
    for df in [train, test]:
        df['장기난임_플래그']      = (df[COL_YEARS].fillna(0) >= 3).astype(int)
        df['나이_시도연수_상호작용'] = df['나이_순서'].fillna(0) * df[COL_YEARS].fillna(0)

print('\n배아 목적 + 고결측 처리 완료')

  임신 시도 또는 마지막 임신 경과 연수: 96.3% 결측 → 플래그 생성
  착상 전 유전 검사 사용 여부: 98.9% 결측 → 플래그 생성
  착상 전 유전 진단 사용 여부: 2.5% 결측 → 플래그 생성
  PGS 시술 여부: 99.2% 결측 → 플래그 생성
  PGD 시술 여부: 99.1% 결측 → 플래그 생성
  난자 해동 경과일: 97.0% 결측 → 플래그 생성

배아 목적 + 고결측 처리 완료


In [14]:
# 범주형 최종 정리
cat_cols = [
    c for c in train.columns
    if c not in [TARGET, ID_COL] and train[c].dtype == 'object'
]
for df in [train, test]:
    for col in cat_cols:
        df[col] = df[col].fillna('missing')
print(f'범주형 결측 처리: {len(cat_cols)}개')

범주형 결측 처리: 0개


In [15]:
# 최종 검증 + Leakage 자동 경보
# 정합성 확인
only_train = set(train.columns) - {TARGET} - set(test.columns)
only_test  = set(test.columns) - set(train.columns)
print('✅ Train/Test 정합성 OK' if not only_train and not only_test
      else f'⚠️ 불일치: train={only_train}, test={only_test}')

# 신규 파생 피처 수
original_cols = set(pd.read_csv('train_clean.csv', nrows=0).columns)
new_cols = [c for c in train.columns if c not in original_cols and c != TARGET]
print(f'✅ 신규 파생 피처: {len(new_cols)}개')
print(f'   전체 피처: {len(train.columns)-2}개 (ID, TARGET 제외)')

✅ Train/Test 정합성 OK
✅ 신규 파생 피처: 103개
   전체 피처: 190개 (ID, TARGET 제외)


In [16]:
# Leakage 자동 경보: 타겟과 상관 0.5↑ 피처 탐지
num_new = [
    c for c in new_cols
    if pd.api.types.is_numeric_dtype(train[c])
]

if TARGET in train.columns:

    corr = train[num_new + [TARGET]].corr(numeric_only=True)[TARGET].drop(TARGET)
    corr_abs = corr.abs().sort_values(ascending=False)

    print('=== 신규 피처 타겟 상관 Top 20 ===')
    display(corr_abs.head(20).round(4))

    suspect = corr_abs[corr_abs > 0.5]

    if len(suspect) > 0:
        print(f'\n🚨 Leakage 의심 (상관 0.5↑): {suspect.index.tolist()}')
        print('   → 데이터 명세서 재확인 필요')
    else:
        print('\n✅ Leakage 의심 변수 없음')

=== 신규 피처 타겟 상관 Top 20 ===


이식배아_있음          0.2443
배양기간_D5추정        0.2278
이식된 배아 수_log     0.1978
배양_기간_일          0.1847
채취_이식_총기간        0.1842
총 생성 배아 수_log    0.1714
나이_성공률_사전점수      0.1547
나이_순서            0.1502
자가난자_고령_조합       0.1453
고위험_3중_조합        0.1453
나이_38이상          0.1411
현재시술용_포함         0.1376
즉시임신단독_여부        0.1372
혼합된 난자 수_log     0.1364
난임_난이도_점수        0.1353
배아저장_있음          0.1305
나이_40이상          0.1299
나이_IVF_상호작용      0.1294
수정_효율            0.1290
잔여배아_수           0.1284
Name: 임신 성공 여부, dtype: float64


✅ Leakage 의심 변수 없음


In [17]:
train.to_csv('train_fe_v3.csv', index=False, encoding='utf-8-sig')
test.to_csv('test_fe_v3.csv',   index=False, encoding='utf-8-sig')
print('저장 완료: train_fe_v3.csv / test_fe_v3.csv')

저장 완료: train_fe_v3.csv / test_fe_v3.csv
